# Phase 3B: Transfer Entropy - Non-Linear Information Flow

## Objective

Quantify the **directional, non-linear information flow** from news sentiment to stock returns that linear Granger Causality cannot detect.

## Core Hypothesis

**High Beta stocks** act as **"Information Sinks"** —> they absorb news sentiment at a significantly higher rate than **Low Beta stocks**, which are driven by fundamentals rather than sentiment.

## Transfer Entropy vs Granger Causality

| Aspect | Granger Causality | Transfer Entropy |
|--------|------------------|------------------|
| **Type** | Linear regression | Information theory |
| **Detects** | Linear predictive relationships | Non-linear information flow |
| **Assumption** | Gaussian errors, linear dynamics | None (model-free) |
| **Captures** | Smooth, gradual effects | Abrupt shocks, regime changes |

---

## Mathematical Foundation

**Transfer Entropy** (Schreiber, 2000):

$$TE_{X \to Y} = H(Y_t | Y_{t-1}) - H(Y_t | Y_{t-1}, X_{t-1})$$

Where:
- $H(Y_t | Y_{t-1})$: Entropy of future returns given past returns (baseline uncertainty)
- $H(Y_t | Y_{t-1}, X_{t-1})$: Entropy of future returns given past returns AND past sentiment
- **Interpretation**: How many bits of information does knowing yesterday's sentiment provide about today's return?

**Directionality**:
- **Forward TE** ($Sentiment \to Price$): News drives price
- **Backward TE** ($Price \to Sentiment$): Price influences news (reflexivity)
- **Net TE** = Forward - Backward: Net information flow

---

## 1. Setup: Imports and Paths

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import entropy, pearsonr
from scipy.special import rel_entr

import matplotlib.pyplot as plt
import seaborn as sns

import os
import random
from tqdm import tqdm
from itertools import product

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.6f}'.format)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

np.random.seed(42)
random.seed(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
project_root = '/content/drive/MyDrive/market-sentiment-impact-analysis'
data_processed = os.path.join(project_root, 'data', 'processed')
plots = os.path.join(project_root, 'plots')

print(f'Project Root: {project_root}')
print(f'Processed Data: {data_processed}')
print(f'Plots: {plots}')

In [ ]:
N_STATES = 3          # 0=Bearish/Negative, 1=Neutral, 2=Bullish/Positive
LAG_K = 1
N_SURROGATES = 200    # num of shuffles for significance testing
ALPHA = 0.05
MIN_OBS = 100

print('Transfer Entropy Configuration:')
print(f'  States: {N_STATES} (Discrete market regimes)')
print(f'  Lag: {LAG_K} trading day(s)')
print(f'  Surrogate tests: {N_SURROGATES}')
print(f'  Significance: α = {ALPHA}')
print(f'  Min observations: {MIN_OBS}')